# Gathering Top Keywords per Topic
<!-- structured-notebook -->
## Notebook Summary
Purpose: extract top c-TF-IDF keywords from each platform's BERTopic model. These keyword lists support topic-matching, paper tables, and manual review.

Main steps:
- Load each platform's BERTopic model.
- Call `get_topic_info()` and extract the top-N keywords per topic.
- Write per-platform `*_top_keywords.csv` files used elsewhere in the pipeline.


# Inputs / Outputs
<!-- io-table -->

| Role | File | Upstream / downstream |
|---|---|---|
| Input | Each platform's BERTopic model directory | Produced by that platform's topic-modeling notebook |
| Output | `<DATA_ROOT>/topic_matchings/<platform>_top_keywords.csv` | Downstream use in paper tables |


Top keywords from Reddit and PubMed topic models

In [ ]:
from src.shared_reddit_telegram.config import (
    PUBMED_MODELS,
)
from collections import Counter
from bertopic import BERTopic
import pandas as pd

# ---- Load your models ----
reddit_model = BERTopic.load("../topic_modelling_v2/round_10/bertopic_no_embed")
pubmed_model = BERTopic.load(PUBMED_MODELS / "10000_docs_all-mpnet-2/BERTopic_model/bertopic_model")

# ---- Get topic representations ----
reddit_topics = reddit_model.get_topics()
pubmed_topics = pubmed_model.get_topics()

def extract_keywords(topics_dict, top_n_words=30):
    """
    Collects and counts how often keywords reoccur across all topics.
    """
    all_terms = []
    for topic_id, words_scores in topics_dict.items():
        if topic_id == -1:  # skip outlier topic
            continue
        all_terms.extend([w for w, _ in words_scores[:top_n_words]])
    return Counter(all_terms)

# ---- Count keyword frequencies ----
reddit_keywords = extract_keywords(reddit_topics)
pubmed_keywords = extract_keywords(pubmed_topics)

# ---- Convert to DataFrame for better readability ----
df_reddit = pd.DataFrame(reddit_keywords.most_common(50), columns=["keyword", "count"])
df_pubmed = pd.DataFrame(pubmed_keywords.most_common(50), columns=["keyword", "count"])

# ---- Save or visualize ----
df_reddit.to_csv("reddit_top_keywords.csv", index=False)
df_pubmed.to_csv("pubmed_top_keywords.csv", index=False)

print("Top 20 Reddit keywords:")
print(df_reddit.head(20))
print("\nTop 20 PubMed keywords:")
print(df_pubmed.head(20))

Co-occuring keywords from PubMed->Reddit and PubMed->Youtube pairings

In [ ]:
# JUPYTER-CELL: Common keywords across PubMed↔Reddit and PubMed↔YouTube pairing CSVs
# Edit these three lines, then run the cell.
PUBMED_REDDIT_CSV  = "../topic-matching/pubmed_simple-reddit/round_3/topic-pairs/keyword_pairs_2.csv"     # e.g., "/mnt/data/keyword_pairs_2.csv"
PUBMED_YOUTUBE_CSV = "../topic-matching/pubmed_simple-youtube/round_3/topic-pairs/keyword_pairs_2.csv"    # another CSV with similar structure
OUT_PREFIX         = "pubmed_reddit_youtube_overlap"       # folder or filename prefix (folder auto-created)
SAVE_CSV           = True                                    # set False to skip writing CSVs

# -------------------------------------------------------------------------
import os, re
from typing import List, Optional, Tuple, Dict
import pandas as pd
import numpy as np
from IPython.display import display

def normalize_token(t: str) -> str:
    if pd.isna(t):
        return ""
    t = str(t).lower().strip()
    t = re.sub(r"[^\w\s-]", "", t)      # keep letters, digits, underscore, whitespace, hyphen
    t = re.sub(r"\s+", " ", t)
    return t

def autodetect_col(df: pd.DataFrame, options: List[str]) -> Optional[str]:
    cols = {c.lower(): c for c in df.columns}
    for cand in options:
        if cand.lower() in cols:
            return cols[cand.lower()]
    return None

def expand_keywords_column(df: pd.DataFrame, kw_col: str) -> pd.DataFrame:
    """
    If keywords are packed in one cell (e.g., "a, b; c|d"), split on common delimiters and explode.
    If they’re already one-per-row, nothing changes.
    """
    sep_pattern = re.compile(r"[;,|/]+")
    ser = df[kw_col].dropna().astype(str)
    if ser.str.contains(sep_pattern).any():
        df = df.copy()
        df[kw_col] = df[kw_col].astype(str).str.split(sep_pattern)
        df = df.explode(kw_col)
        df[kw_col] = df[kw_col].astype(str).str.strip()
        df = df[df[kw_col] != ""]
        df = df.reset_index(drop=True)
    return df

def load_and_prepare(
    path: str,
    pubmed_col_opts=("pubmed_simple_topic_num","pubmed_topic","topic_a","pubmed","pubmedId"),
    other_col_opts=("reddit_topic_num","youtube_topic_num","other_topic","topic_b","other","pair"),
    kw_col_opts=("common_terms","co_token","keyword","term","token","kw","word","phrase","norm","keywords"),
    score_col_opts=("coocc_score","score","weight","value","strength"),
) -> Tuple[pd.DataFrame, str, Optional[str], str, Optional[str]]:
    """
    Returns (df, pubmed_col, other_col_or_None, kw_col, score_col_or_None)
    - normalizes keywords into 'kw_norm'
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    df = pd.read_csv(path)

    pubmed_col = autodetect_col(df, list(pubmed_col_opts))
    if not pubmed_col:
        raise ValueError(f"Could not detect PubMed topic column in {path}. "
                         f"Tried {list(pubmed_col_opts)}")

    other_col = autodetect_col(df, list(other_col_opts))  # optional (only needed for strict triads)

    kw_col = autodetect_col(df, list(kw_col_opts))
    if not kw_col:
        raise ValueError(f"Could not detect keyword column in {path}. "
                         f"Tried {list(kw_col_opts)}")

    score_col = autodetect_col(df, list(score_col_opts))  # optional

    # explode list-like cells if needed
    df = expand_keywords_column(df, kw_col)

    # normalize tokens
    df["kw_norm"] = df[kw_col].astype(str).map(normalize_token)
    df = df[df["kw_norm"] != ""].copy().reset_index(drop=True)

    return df, pubmed_col, other_col, kw_col, score_col

def aggregate_strength(df: pd.DataFrame, by_cols: List[str], score_col: Optional[str]) -> pd.DataFrame:
    """
    If a score column exists, sum it; otherwise count occurrences.
    Output column is 'strength'.
    """
    if score_col and score_col in df.columns:
        agg = (df.groupby(by_cols, as_index=False)[score_col]
                 .sum()
                 .rename(columns={score_col: "strength"}))
    else:
        agg = (df.groupby(by_cols, as_index=False)
                 .size()
                 .rename(columns={"size": "strength"}))
    return agg

def run_common_keyword_analysis(
    pubmed_reddit_csv: str,
    pubmed_youtube_csv: str,
    out_prefix: str = ".",
    save_csv: bool = True,
) -> Dict[str, pd.DataFrame]:
    """
    Computes:
      - Global overlap across both pairing sets
      - Per-PubMed-topic overlap
      - (Optional) strict triad overlap if both CSVs include paired other-topic IDs
    Returns a dict of DataFrames (also writes CSVs if save_csv=True).
    """
    out_dir = os.path.dirname(out_prefix)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)

    # Load both pairing tables (each already contains keywords)
    pr, pr_pubmed, pr_other, pr_kw, pr_score = load_and_prepare(pubmed_reddit_csv)
    py, py_pubmed, py_other, py_kw, py_score = load_and_prepare(pubmed_youtube_csv)

    # ---------- 1) GLOBAL OVERLAP ----------
    pr_kw_set = set(pr["kw_norm"].unique())
    py_kw_set = set(py["kw_norm"].unique())
    common_global = sorted(pr_kw_set & py_kw_set)

    pr_glob_strength = aggregate_strength(pr, ["kw_norm"], pr_score).rename(columns={"strength":"pubmed_reddit_strength"})
    py_glob_strength = aggregate_strength(py, ["kw_norm"], py_score).rename(columns={"strength":"pubmed_youtube_strength"})

    global_df = pd.DataFrame({"kw_norm": common_global})
    global_df = (global_df
                 .merge(pr_glob_strength, on="kw_norm", how="left")
                 .merge(py_glob_strength, on="kw_norm", how="left"))
    global_df[["pubmed_reddit_strength","pubmed_youtube_strength"]] = global_df[["pubmed_reddit_strength","pubmed_youtube_strength"]].fillna(0)
    global_df["combined_strength"] = global_df["pubmed_reddit_strength"] + global_df["pubmed_youtube_strength"]
    global_df = global_df.sort_values("combined_strength", ascending=False).reset_index(drop=True)

    if save_csv:
        global_df.to_csv(f"{out_prefix}_keywords_global.csv", index=False)

    # ---------- 2) PER PUBMED TOPIC ----------
    pr_per_pubmed = pr.groupby(pr_pubmed)["kw_norm"].apply(set).reset_index(name="pr_kw_set")
    py_per_pubmed = py.groupby(py_pubmed)["kw_norm"].apply(set).reset_index(name="py_kw_set")

    per_pubmed = pr_per_pubmed.merge(py_per_pubmed, left_on=pr_pubmed, right_on=py_pubmed, how="inner")
    per_pubmed["pubmed_topic"] = per_pubmed[pr_pubmed]
    per_pubmed["common_kw"] = per_pubmed.apply(lambda r: sorted(r["pr_kw_set"] & r["py_kw_set"]), axis=1)
    per_pubmed_long = per_pubmed.explode("common_kw").dropna(subset=["common_kw"]).copy()
    per_pubmed_long.rename(columns={"common_kw":"kw_norm"}, inplace=True)

    pr_str = aggregate_strength(pr, [pr_pubmed, "kw_norm"], pr_score).rename(columns={"strength":"pubmed_reddit_strength"})
    py_str = aggregate_strength(py, [py_pubmed, "kw_norm"], py_score).rename(columns={"strength":"pubmed_youtube_strength"})

    per_pubmed_out = (per_pubmed_long[["pubmed_topic","kw_norm"]]
        .merge(pr_str, left_on=["pubmed_topic","kw_norm"], right_on=[pr_pubmed,"kw_norm"], how="left")
        .drop(columns=[pr_pubmed])
        .merge(py_str, left_on=["pubmed_topic","kw_norm"], right_on=[py_pubmed,"kw_norm"], how="left")
        .drop(columns=[py_pubmed])
    )
    per_pubmed_out[["pubmed_reddit_strength","pubmed_youtube_strength"]] = per_pubmed_out[["pubmed_reddit_strength","pubmed_youtube_strength"]].fillna(0)
    per_pubmed_out["combined_strength"] = per_pubmed_out["pubmed_reddit_strength"] + per_pubmed_out["pubmed_youtube_strength"]
    per_pubmed_out = per_pubmed_out.sort_values(["pubmed_topic","combined_strength"], ascending=[True, False]).reset_index(drop=True)

    if save_csv:
        per_pubmed_out.to_csv(f"{out_prefix}_keywords_per_pubmed.csv", index=False)

    # ---------- 3) STRICT SAME-PAIR OVERLAP (triads) ----------
    # Only computed if BOTH CSVs include the paired other-topic ID columns (e.g., reddit_topic_num, youtube_topic_num)
    strict_df = pd.DataFrame(columns=[
        "pubmed_topic","reddit_topic","youtube_topic","kw_norm",
        "pubmed_reddit_strength","pubmed_youtube_strength","combined_strength"
    ])
    if pr_other and py_other:
        pr_pairs = pr.groupby([pr_pubmed, pr_other])["kw_norm"].apply(set).reset_index(name="pr_kw_set")
        py_pairs = py.groupby([py_pubmed, py_other])["kw_norm"].apply(set).reset_index(name="py_kw_set")

        merged = pr_pairs.merge(py_pairs, left_on=pr_pubmed, right_on=py_pubmed, suffixes=("_pr","_py"))
        merged["common_kw"] = merged.apply(lambda r: sorted(r["pr_kw_set"] & r["py_kw_set"]), axis=1)
        merged = merged[merged["common_kw"].map(len) > 0].copy()

        pr_pair_strength = aggregate_strength(pr, [pr_pubmed, pr_other, "kw_norm"], pr_score).rename(columns={"strength":"pubmed_reddit_strength"})
        py_pair_strength = aggregate_strength(py, [py_pubmed, py_other, "kw_norm"], py_score).rename(columns={"strength":"pubmed_youtube_strength"})

        strict_df = (merged.explode("common_kw")
                     .rename(columns={
                         pr_pubmed: "pubmed_topic",
                         pr_other: "reddit_topic",
                         py_other: "youtube_topic",
                         "common_kw": "kw_norm"
                     })[["pubmed_topic","reddit_topic","youtube_topic","kw_norm"]])

        strict_df = (strict_df
            .merge(pr_pair_strength, left_on=["pubmed_topic","reddit_topic","kw_norm"],
                   right_on=[pr_pubmed, pr_other, "kw_norm"], how="left")
            .drop(columns=[pr_pubmed, pr_other])
            .merge(py_pair_strength, left_on=["pubmed_topic","youtube_topic","kw_norm"],
                   right_on=[py_pubmed, py_other, "kw_norm"], how="left")
            .drop(columns=[py_pubmed, py_other])
        )
        strict_df[["pubmed_reddit_strength","pubmed_youtube_strength"]] = strict_df[["pubmed_reddit_strength","pubmed_youtube_strength"]].fillna(0)
        strict_df["combined_strength"] = strict_df["pubmed_reddit_strength"] + strict_df["pubmed_youtube_strength"]
        strict_df = strict_df.sort_values(["pubmed_topic","combined_strength"], ascending=[True, False]).reset_index(drop=True)

    if save_csv:
        strict_df.to_csv(f"{out_prefix}_keywords_strict_pairs.csv", index=False)

    # Console summary
    print(f"[OK] Global overlap keywords: {len(global_df)}")
    print(f"[OK] Per-PubMed overlap rows: {len(per_pubmed_out)}")
    if pr_other and py_other:
        print(f"[OK] Strict triad overlap rows: {len(strict_df)}")
    else:
        print("[INFO] Strict triad overlap skipped (missing paired other-topic id column in one or both CSVs).")

    return {
        "global": global_df,
        "per_pubmed": per_pubmed_out,
        "strict_pairs": strict_df
    }

# ---- Run it (uses the paths you set at the top) ----
results = run_common_keyword_analysis(
    pubmed_reddit_csv=PUBMED_REDDIT_CSV,
    pubmed_youtube_csv=PUBMED_YOUTUBE_CSV,
    out_prefix=OUT_PREFIX,
    save_csv=SAVE_CSV
)

# Quick peek
display(results["global"].head(20))
display(results["per_pubmed"].head(20))
if not results["strict_pairs"].empty:
    display(results["strict_pairs"].head(20))

---
<!-- nav-footer -->

← [04 topic doc exploration](../../../../../SocialMedia/Reddit/notebooks/BERTopic/04_topic_matching/04_topic_doc_exploration.ipynb) | [Project Overview](../../../../../PROJECT_OVERVIEW.ipynb) | [06 pkl csv conversion](../../../../../SocialMedia/Reddit/notebooks/BERTopic/04_topic_matching/06_pkl_csv_conversion.ipynb) →
